<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/frtb_curvature_notebook_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- Step 3 & 4: Establish Gross and Net Positions ---

# Create a DataFrame with the gross positions from the report.
gross_positions_data = {
    'Bucket': [10, 10, 2, 2],
    'Product': ['Sugar', 'Sugar', 'Brent', 'WTI'],
    'CVR+': [26177, 91620, 124433, -35460],
    'CVR-': [14197, 49690, -152185, 63068]
}
gross_df = pd.DataFrame(gross_positions_data)

print("--- Step 3: Gross Positions ---")
print(gross_df)
print("\n" + "="*50 + "\n")

# Per Article 325g(1), we net positions for the same risk factor.
# For commodity curvature, 'Sugar' is one risk factor, 'Brent' another, and 'WTI' a third.
# We group by 'Bucket' and 'Product' and sum the CVRs to get the net positions.
net_df = gross_df.groupby(['Bucket', 'Product']).sum().reset_index()

print("--- Step 4: Net Positions ---")
print("Netting the two 'Sugar' positions as they belong to the same risk factor.")
print(net_df)
print("\n" + "="*50 + "\n")


# --- Step 5 & 6: Determine Correlation Parameters (Medium Scenario) ---

# Intra-bucket delta correlation for Bucket 2 (Brent vs WTI) - Article 325at(2A)
# Commodity type (95%) * Tenor (100%) * Basis (100%, ignored for curvature per 325p(4))
delta_rho_kl = 0.95 * 1.0 * 1.0

# Cross-bucket delta correlation for Bucket 2 vs Bucket 10 - Article 325au(a)
delta_gamma_bc = 0.20

# Curvature correlations are the square of delta correlations - Article 325ay(5)
curvature_rho_kl_medium = delta_rho_kl**2
curvature_gamma_bc_medium = delta_gamma_bc**2

print("--- Step 5 & 6: Correlation Parameters (Medium Scenario) ---")
print(f"Intra-Bucket Delta Correlation (rho_kl): {delta_rho_kl:.2%}")
print(f"Intra-Bucket Curvature Correlation (rho_kl^2): {curvature_rho_kl_medium:.2%}\n")
print(f"Cross-Bucket Delta Correlation (gamma_bc): {delta_gamma_bc:.2%}")
print(f"Cross-Bucket Curvature Correlation (gamma_bc^2): {curvature_gamma_bc_medium:.2%}")
print("\n" + "="*50 + "\n")


# --- Step 8: Calculate Bucket-Level Capital (Medium Scenario) ---

def calculate_bucket_capital(cvr_data, intra_bucket_corr):
    """Calculates Kb+, Kb-, and the final Kb for a given bucket."""

    # Safeguard function psi
    def psi(x, y):
        return 0 if x < 0 and y < 0 else 1

    # Isolate CVR+ and CVR- values for the bucket
    cvr_plus = cvr_data['CVR+'].values
    cvr_minus = cvr_data['CVR-'].values

    # Calculate Kb+ (Upward Scenario)
    sum_max_cvr_plus_sq = np.sum(np.maximum(cvr_plus, 0)**2)

    correlation_term_plus = 0
    if len(cvr_plus) > 1:
        # Simplified for 2 risk factors
        correlation_term_plus = 2 * intra_bucket_corr * cvr_plus[0] * cvr_plus[1] * psi(cvr_plus[0], cvr_plus[1])

    kb_plus = np.sqrt(max(0, sum_max_cvr_plus_sq + correlation_term_plus))

    # Calculate Kb- (Downward Scenario)
    sum_max_cvr_minus_sq = np.sum(np.maximum(cvr_minus, 0)**2)

    correlation_term_minus = 0
    if len(cvr_minus) > 1:
        correlation_term_minus = 2 * intra_bucket_corr * cvr_minus[0] * cvr_minus[1] * psi(cvr_minus[0], cvr_minus[1])

    kb_minus = np.sqrt(max(0, sum_max_cvr_minus_sq + correlation_term_minus))

    # Determine final Kb and selected scenario
    if kb_plus > kb_minus:
        kb_final = kb_plus
        scenario = "Upward"
    elif kb_minus > kb_plus:
        kb_final = kb_minus
        scenario = "Downward"
    else: # If they are equal, choose based on the larger sum of CVRs
        kb_final = kb_plus
        scenario = "Upward" if np.sum(cvr_plus) >= np.sum(cvr_minus) else "Downward"

    return kb_plus, kb_minus, kb_final, scenario

print("--- Step 8: Bucket-Level Capital (Medium Scenario) ---")

# Bucket 10 (Sugar)
b10_data = net_df[net_df['Bucket'] == 10]
kb10_plus, kb10_minus, kb10_final, scenario10 = calculate_bucket_capital(b10_data, 0) # No intra-bucket corr needed
print("Bucket 10 (Sugar):")
print(f"  - K_b+ = {kb10_plus:,.0f}")
print(f"  - K_b- = {kb10_minus:,.0f}")
print(f"  - Final K_b = {kb10_final:,.0f} (Selected Scenario: {scenario10})\n")

# Bucket 2 (Brent & WTI)
b2_data = net_df[net_df['Bucket'] == 2]
kb2_plus, kb2_minus, kb2_final, scenario2 = calculate_bucket_capital(b2_data, curvature_rho_kl_medium)
print("Bucket 2 (Brent & WTI):")
print(f"  - K_b+ = {kb2_plus:,.0f}")
print(f"  - K_b- = {kb2_minus:,.0f}")
print(f"  - Final K_b = {kb2_final:,.0f} (Selected Scenario: {scenario2})")
print("\n" + "="*50 + "\n")


# --- Step 9: Determine Bucket Sums (Medium Scenario) ---

def calculate_bucket_sum(cvr_data, scenario):
    """Calculates Sb based on the selected scenario."""
    if scenario == "Upward":
        return cvr_data['CVR+'].sum()
    else:
        return cvr_data['CVR-'].sum()

s10 = calculate_bucket_sum(b10_data, scenario10)
s2 = calculate_bucket_sum(b2_data, scenario2)

print("--- Step 9: Bucket Sums (Medium Scenario) ---")
print(f"Bucket 10 Sum (S_b): {s10:,.0f} (from {scenario10} scenario)")
print(f"Bucket 2 Sum (S_b): {s2:,.0f} (from {scenario2} scenario)")
print("\n" + "="*50 + "\n")


# --- Step 10: Cross-Bucket Capital (Medium Scenario) ---

def calculate_rccr(bucket_capitals, bucket_sums, cross_bucket_corr):
    """Calculates the final Risk Class Curvature Requirement (RCCR)."""

    def psi(x, y):
        return 0 if x < 0 and y < 0 else 1

    sum_kb_sq = np.sum(np.array(bucket_capitals)**2)

    correlation_term = 0
    if len(bucket_sums) > 1:
        # Simplified for 2 buckets
        correlation_term = 2 * cross_bucket_corr * bucket_sums[0] * bucket_sums[1] * psi(bucket_sums[0], bucket_sums[1])

    rccr = np.sqrt(max(0, sum_kb_sq + correlation_term))
    return rccr

rccr_medium = calculate_rccr([kb10_final, kb2_final], [s10, s2], curvature_gamma_bc_medium)

print("--- Step 10: Cross-Bucket Capital (Medium Scenario) ---")
print(f"Total Capital (RCCR) for Medium Scenario: £{rccr_medium:,.0f}")
print("\n" + "="*50 + "\n")


# --- Step 11: All Scenarios and Final Capital ---

print("--- Step 11: All Scenarios and Final Capital ---")

scenarios = {
    "High": {"rho_delta": min(delta_rho_kl * 1.25, 1.0), "gamma_delta": min(delta_gamma_bc * 1.25, 1.0)},
    "Low": {"rho_delta": max(2 * delta_rho_kl - 1, 0.75 * delta_rho_kl), "gamma_delta": max(2 * delta_gamma_bc - 1, 0.75 * delta_gamma_bc)},
    "Medium": {"rho_delta": delta_rho_kl, "gamma_delta": delta_gamma_bc}
}

results = []

for name, params in scenarios.items():
    # Get scenario-specific curvature correlations
    rho_curv = params["rho_delta"]**2
    gamma_curv = params["gamma_delta"]**2

    # Recalculate Bucket 2 capital with new intra-bucket correlation
    _, _, kb2_scen, scen2_scen = calculate_bucket_capital(b2_data, rho_curv)

    # Bucket sums depend on the new scenario selection for Bucket 2
    s2_scen = calculate_bucket_sum(b2_data, scen2_scen)

    # Recalculate total RCCR
    rccr_scen = calculate_rccr([kb10_final, kb2_scen], [s10, s2_scen], gamma_curv)

    results.append({
        "Scenario": name,
        "Intra-Bucket Curv. Corr.": rho_curv,
        "Cross-Bucket Curv. Corr.": gamma_curv,
        "Bucket 2 Capital (K_2)": kb2_scen,
        "Total Capital (RCCR)": rccr_scen
    })

results_df = pd.DataFrame(results).sort_values("Scenario", key=lambda x: x.map({"Low":0, "Medium":1, "High":2})).set_index("Scenario")

print("Summary of Capital Across All Scenarios:")
print(results_df.to_string(formatters={
    'Intra-Bucket Curv. Corr.': '{:,.2%}'.format,
    'Cross-Bucket Curv. Corr.': '{:,.2%}'.format,
    'Bucket 2 Capital (K_2)': '£{:,.0f}'.format,
    'Total Capital (RCCR)': '£{:,.0f}'.format
}))

final_capital = results_df['Total Capital (RCCR)'].max()

print("\n" + "-"*30)
print(f"Final Commodity Curvature Capital Requirement: £{final_capital:,.0f}")
print("-" * 30)

--- Step 3: Gross Positions ---
   Bucket Product    CVR+    CVR-
0      10   Sugar   26177   14197
1      10   Sugar   91620   49690
2       2   Brent  124433 -152185
3       2     WTI  -35460   63068


--- Step 4: Net Positions ---
Netting the two 'Sugar' positions as they belong to the same risk factor.
   Bucket Product    CVR+    CVR-
0       2   Brent  124433 -152185
1       2     WTI  -35460   63068
2      10   Sugar  117797   63887


--- Step 5 & 6: Correlation Parameters (Medium Scenario) ---
Intra-Bucket Delta Correlation (rho_kl): 95.00%
Intra-Bucket Curvature Correlation (rho_kl^2): 90.25%

Cross-Bucket Delta Correlation (gamma_bc): 20.00%
Cross-Bucket Curvature Correlation (gamma_bc^2): 4.00%


--- Step 8: Bucket-Level Capital (Medium Scenario) ---
Bucket 10 (Sugar):
  - K_b+ = 117,797
  - K_b- = 63,887
  - Final K_b = 117,797 (Selected Scenario: Upward)

Bucket 2 (Brent & WTI):
  - K_b+ = 86,713
  - K_b- = 0
  - Final K_b = 86,713 (Selected Scenario: Upward)


--- Step 9: